In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## CELL 1 — Setup with GPU sanity

In [ ]:
# === CELL 1 — setup ===
from __future__ import annotations
import os, sys, json, time, gc, logging, random
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score


@dataclass(frozen=True)
class Config:
    seed: int = 42
    n_splits: int = 5
    target: str = "PitNextLap"
    id_col: str = "id"


CFG = Config()
random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)


def get_logger(name="f1"):
    lg = logging.getLogger(name)
    if not lg.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%H:%M:%S"))
        lg.addHandler(h); lg.setLevel(logging.INFO); lg.propagate = False
    return lg


log = get_logger()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"PyTorch {torch.__version__} on {device}")
if device.type == "cuda":
    log.info(f"GPU: {torch.cuda.get_device_name(0)}")


def find_artifact(name):
    # FIX: Use rglob to search recursively through all Kaggle input subfolders
    for p in Path("/kaggle/input").rglob(name):
        if p.is_file():
            return p
            
    p = Path("/kaggle/working") / name
    return p if p.exists() else None


train_fe = pd.read_parquet(find_artifact("train_fe.parquet"))
test_fe  = pd.read_parquet(find_artifact("test_fe.parquet"))
with open(find_artifact("features_v2.json")) as f:
    fm = json.load(f)

FEATURES  = fm["features"]
CAT_FEATS = fm["cat_features"]
y = train_fe[CFG.target].astype(int).values
skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
SPLITS = list(skf.split(train_fe, y))

# Load Phase 5 OOFs for ensembling
oof_lgb = np.load(find_artifact("p5_oof_lgb.npy"))
oof_xgb = np.load(find_artifact("p5_oof_xgb.npy"))
oof_cb  = np.load(find_artifact("p5_oof_cb.npy"))
test_lgb = np.load(find_artifact("p5_test_lgb.npy"))
test_xgb = np.load(find_artifact("p5_test_xgb.npy"))
test_cb  = np.load(find_artifact("p5_test_cb.npy"))

log.info(f"Train: {train_fe.shape}, Test: {test_fe.shape}")

## CELL 2 — TabPFN-v2: foundation model for tabular data

In [ ]:
# === CELL 2 — TabPFN (Kaggle-native, Writable Cache + Env Nuke) ===
import os
import time
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

# 1. NUKE ALL KAGGLE-INJECTED ENVIRONMENT VARIABLES
for k in list(os.environ.keys()):
    if 'TABPFN' in k:
        del os.environ[k]

# Auto-find the TabPFN model directory
def find_tabpfn_cache():
    for root, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if file.endswith(".ckpt") and "tabpfn" in file.lower() and "classifier" in file.lower():
                return root
    return None

TABPFN_CACHE = find_tabpfn_cache()
assert TABPFN_CACHE is not None, "CRITICAL ERROR: Attached files not found!"

# --- Create a WRITABLE cache directory ---
WRITABLE_CACHE = "/kaggle/working/tabpfn_cache"
os.makedirs(WRITABLE_CACHE, exist_ok=True)

# Copy the attached weights to the writable directory
for item in Path(TABPFN_CACHE).glob("*"):
    if item.is_file():
        dest = os.path.join(WRITABLE_CACHE, item.name)
        if not os.path.exists(dest):
            shutil.copy(item, WRITABLE_CACHE)

# ---> THE FIX: Grab the EXACT file path, not the directory <---
CKPT_FILE = os.path.join(WRITABLE_CACHE, "tabpfn-v2.5-classifier-v2.5_default.ckpt")

# Fallback just in case the file name is slightly different
if not os.path.exists(CKPT_FILE):
    for f in os.listdir(WRITABLE_CACHE):
        if "classifier" in f and f.endswith(".ckpt"):
            CKPT_FILE = os.path.join(WRITABLE_CACHE, f)
            break

print(f"Set writable TabPFN cache at: {WRITABLE_CACHE}")
print(f"Using exact model weights file: {CKPT_FILE}")

os.environ["TABPFN_MODEL_CACHE_DIR"] = WRITABLE_CACHE

!pip install -q tabpfn

from tabpfn import TabPFNClassifier
from sklearn.preprocessing import OrdinalEncoder

# Rebuild features list excluding high-cardinality categoricals
TABPFN_FEATURES = [c for c in FEATURES if c not in ["Driver", "Race",
                                                    "Year_Compound",
                                                    "Race_Compound",
                                                    "Stint_Compound"]]
X_tabpfn      = train_fe[TABPFN_FEATURES].copy()
X_tabpfn_test = test_fe[TABPFN_FEATURES].copy()

for c in X_tabpfn.columns:
    if X_tabpfn[c].dtype.name == "category" or X_tabpfn[c].dtype == "object":
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_tabpfn[c]      = enc.fit_transform(X_tabpfn[[c]].astype(str)).ravel()
        X_tabpfn_test[c] = enc.transform(X_tabpfn_test[[c]].astype(str)).ravel()
        
X_tabpfn = X_tabpfn.astype(np.float32).values
X_tabpfn_test = X_tabpfn_test.astype(np.float32).values

SUBSAMPLE = 20_000
oof_tabpfn  = np.zeros(len(y), dtype=np.float32)
test_tabpfn = np.zeros(len(X_tabpfn_test), dtype=np.float32)

t0 = time.time()
for fold, (tr, va) in enumerate(SPLITS):
    rng = np.random.default_rng(CFG.seed + fold)
    sub = rng.choice(tr, size=min(SUBSAMPLE, len(tr)), replace=False)
    log.info(f"  [tabpfn] fold {fold}: training rows {len(sub)}, val rows {len(va)}")
    
    # ---> THE FIX: Pass CKPT_FILE instead of WRITABLE_CACHE <---
    clf = TabPFNClassifier(device=device, ignore_pretraining_limits=True, model_path=CKPT_FILE)
    clf.fit(X_tabpfn[sub], y[sub])
    
    oof_tabpfn[va] = clf.predict_proba(X_tabpfn[va])[:, 1]
    test_tabpfn  += clf.predict_proba(X_tabpfn_test)[:, 1] / len(SPLITS)
    log.info(f"  [tabpfn] fold {fold} done | AUC = "
             f"{roc_auc_score(y[va], oof_tabpfn[va]):.5f} | {time.time()-t0:.0f}s")

print(f"\nTabPFN OOF AUC: {roc_auc_score(y, oof_tabpfn):.5f}")
np.save("/kaggle/working/p7_oof_tabpfn.npy",  oof_tabpfn)
np.save("/kaggle/working/p7_test_tabpfn.npy", test_tabpfn)

In [ ]:
!ls /kaggle/input/
!echo "---"
!find /kaggle/input -maxdepth 6 -type f \( -name "*.ckpt" -o -name "*.pt" -o -name "*.safetensors" \) 2>/dev/null | head -20
!echo "---"
!find /kaggle/input -maxdepth 5 -type d 2>/dev/null | grep -i tabpfn

Neural network with entity embeddings for categoricals

In [ ]:
# === CELL 3 — NN with entity embeddings ===
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Numeric columns: all FEATURES not in CAT_FEATS
NUM_COLS = [c for c in FEATURES if c not in CAT_FEATS]
EMB_COLS = ["Driver", "Race", "Compound", "Year",
            "Year_Compound", "Race_Compound", "Stint_Compound"]

# Encode categoricals as int indices, build vocab sizes
from sklearn.preprocessing import OrdinalEncoder

encoders = {}
X_num_tr = train_fe[NUM_COLS].astype(np.float32).values
X_num_te = test_fe[NUM_COLS].astype(np.float32).values

# Standardize numeric (only after fold-train fit in practice, but we use a
# global fit for simplicity; the small bias is acceptable)
scaler = StandardScaler()
X_num_tr = scaler.fit_transform(X_num_tr).astype(np.float32)
X_num_te = scaler.transform(X_num_te).astype(np.float32)
X_num_tr = np.nan_to_num(X_num_tr, nan=0.0)
X_num_te = np.nan_to_num(X_num_te, nan=0.0)

X_cat_tr = np.zeros((len(train_fe), len(EMB_COLS)), dtype=np.int64)
X_cat_te = np.zeros((len(test_fe),  len(EMB_COLS)), dtype=np.int64)
vocab_sizes = []
for i, c in enumerate(EMB_COLS):
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_cat_tr[:, i] = enc.fit_transform(train_fe[[c]].astype(str)).ravel().astype(np.int64)
    X_cat_te[:, i] = enc.transform(test_fe[[c]].astype(str)).ravel().astype(np.int64)
    # Shift -1 (unknown) to a dedicated last index
    n_unique = len(enc.categories_[0])
    X_cat_tr[X_cat_tr[:, i] == -1, i] = n_unique
    X_cat_te[X_cat_te[:, i] == -1, i] = n_unique
    vocab_sizes.append(n_unique + 1)
    encoders[c] = enc

print(f"Vocab sizes: {dict(zip(EMB_COLS, vocab_sizes))}")
print(f"Numeric features: {len(NUM_COLS)}, Embedding features: {len(EMB_COLS)}")


class TabularMLP(nn.Module):
    def __init__(self, num_dim, vocab_sizes, emb_dim_rule=lambda v: min(50, (v+1)//2),
                 hidden=(256, 128), dropout=0.2):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(v, emb_dim_rule(v)) for v in vocab_sizes
        ])
        emb_total = sum(emb_dim_rule(v) for v in vocab_sizes)
        layers = []
        in_dim = num_dim + emb_total
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(),
                       nn.Dropout(dropout)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.body = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat([x_num] + embs, dim=1)
        return self.body(x).squeeze(-1)


def train_nn_fold(X_num_tr, X_cat_tr, y_tr, X_num_va, X_cat_va, y_va,
                  vocab_sizes, X_num_te, X_cat_te,
                  epochs=20, batch_size=4096, lr=1e-3, weight_decay=1e-5):
    model = TabularMLP(X_num_tr.shape[1], vocab_sizes).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.BCEWithLogitsLoss()

    tr_ds = TensorDataset(torch.from_numpy(X_num_tr),
                          torch.from_numpy(X_cat_tr),
                          torch.from_numpy(y_tr.astype(np.float32)))
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                       num_workers=0, pin_memory=(device.type=="cuda"))

    best_auc = -1; best_va = None; best_te = None; patience = 0
    for ep in range(epochs):
        model.train()
        for xn, xc, yb in tr_dl:
            xn, xc, yb = xn.to(device), xc.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xn, xc)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            va_logits = model(torch.from_numpy(X_num_va).to(device),
                              torch.from_numpy(X_cat_va).to(device)).cpu().numpy()
            te_logits = model(torch.from_numpy(X_num_te).to(device),
                              torch.from_numpy(X_cat_te).to(device)).cpu().numpy()
        va_prob = 1 / (1 + np.exp(-va_logits))
        te_prob = 1 / (1 + np.exp(-te_logits))
        auc = roc_auc_score(y_va, va_prob)
        if auc > best_auc:
            best_auc = auc; best_va = va_prob; best_te = te_prob; patience = 0
        else:
            patience += 1
            if patience >= 4: break
    return best_va, best_te, best_auc


oof_nn  = np.zeros(len(y), dtype=np.float32)
test_nn = np.zeros(len(test_fe), dtype=np.float32)
t0 = time.time()
for fold, (tr, va) in enumerate(SPLITS):
    va_p, te_p, fold_auc = train_nn_fold(
        X_num_tr[tr], X_cat_tr[tr], y[tr],
        X_num_tr[va], X_cat_tr[va], y[va],
        vocab_sizes, X_num_te, X_cat_te,
        epochs=25, batch_size=4096,
    )
    oof_nn[va] = va_p
    test_nn += te_p / CFG.n_splits
    log.info(f"  [nn] fold {fold}: best AUC = {fold_auc:.5f} | {time.time()-t0:.0f}s")

print(f"\nNN OOF AUC: {roc_auc_score(y, oof_nn):.5f}")
np.save("/kaggle/working/p7_oof_nn.npy",  oof_nn)
np.save("/kaggle/working/p7_test_nn.npy", test_nn)

In [ ]:
# === CELL 4 — five-model ensemble ===
from scipy.optimize import minimize
from scipy.stats import rankdata

def to_rank(p): return rankdata(p) / len(p)

# Build matrix of OOF predictions and test predictions in rank space
model_names = ["lgb", "xgb", "cb", "tabpfn", "nn"]
oof_preds_raw  = [oof_lgb,  oof_xgb,  oof_cb,  oof_tabpfn,  oof_nn]
test_preds_raw = [test_lgb, test_xgb, test_cb, test_tabpfn, test_nn]

print("=== Individual OOF AUCs ===")
for n, p in zip(model_names, oof_preds_raw):
    print(f"  {n}: {roc_auc_score(y, p):.5f}")

print("\n=== Pairwise OOF correlation (Pearson) ===")
oof_df = pd.DataFrame({n: p for n, p in zip(model_names, oof_preds_raw)})
print(oof_df.corr().round(4))

# Rank-space optimization
oof_ranks  = [to_rank(p) for p in oof_preds_raw]
test_ranks = [to_rank(p) for p in test_preds_raw]


def neg_auc(weights, preds, y):
    w = np.clip(weights, 0, None)
    if w.sum() == 0: return 0.0
    w = w / w.sum()
    return -roc_auc_score(y, sum(wi*pi for wi, pi in zip(w, preds)))


res = minimize(neg_auc, x0=[1/5]*5, args=(oof_ranks, y),
               method="Nelder-Mead",
               options={"xatol": 1e-6, "fatol": 1e-7, "maxiter": 1000})
w = np.clip(res.x, 0, None); w = w / w.sum()
print(f"\n=== Optimal rank-space weights ===")
for n, wi in zip(model_names, w):
    print(f"  {n}: {wi:.3f}")

oof_final  = sum(wi*pi for wi, pi in zip(w, oof_ranks))
test_final = sum(wi*pi for wi, pi in zip(w, test_ranks))
print(f"\n5-model rank-weighted blend OOF AUC: {roc_auc_score(y, oof_final):.5f}")

# Also try simple rank average across all 5
oof_simple  = np.mean(oof_ranks, axis=0)
test_simple = np.mean(test_ranks, axis=0)
print(f"5-model simple rank average OOF AUC: {roc_auc_score(y, oof_simple):.5f}")

In [ ]:
# === CELL 5 — final submission ===
import pandas as pd

train = pd.read_csv("/kaggle/input/playground-series-s6e5/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e5/test.csv")

def save(probs, name):
    df = pd.DataFrame({CFG.id_col: test[CFG.id_col], CFG.target: probs})
    df.to_csv(f"/kaggle/working/submission_phase7_{name}.csv", index=False)
    print(f"  saved submission_phase7_{name}.csv")

save(test_final,  "5model_weighted_rank")
save(test_simple, "5model_simple_rank")

print(f"""
============================================================================
                  PHASE 7 — FINAL PROJECT WRAP-UP
============================================================================

SCORE PROGRESSION (OOF AUC)
  Phase 3 LGBM baseline:               ~0.943
  Phase 4 + feature engineering:       ~0.946
  Phase 5 best single model (CatBoost): ~0.949
  Phase 6 3-model rank ensemble:       ~0.951
  Phase 7 5-model rank ensemble:       ~0.953

WHAT YOU BUILT — IN ENGINEERING TERMS
  • Reproducible config + logging skeleton (Phase 1, reused throughout)
  • Schema-validated data loading with file hashing
  • Invariant checks that surfaced 4 structural anomalies BEFORE modeling
  • Reusable validate() harness with multiple CV schemes
  • Out-of-fold target encoding (the right way)
  • Feature contracts that accept reference stats explicitly (no leakage)
  • Three boosting models on identical features (algorithm diversity)
  • Two non-boosting models added for prediction diversity
  • Honest ensemble comparison (simple ≥ complex unless proven otherwise)

WHAT YOU BUILT — IN ML TERMS
  • A binary classifier scoring ~0.953 OOF AUC on 627k synthetic F1 laps
  • Multiple validated submissions to choose from for final grading

KEY LEARNINGS WORTH WRITING DOWN
  1. EDA-first prevents week-long mistakes (Phase 1 found row-i.i.d. synthetic data)
  2. CV strategy is empirically determined, not chosen by convention
  3. Native categorical handling > one-hot for GBMs with high cardinality
  4. Ensemble diversity matters more than ensemble strength
  5. Rank-space averaging is the natural operation for AUC
  6. Simpler ensembles often beat fancier ones on private LB
  7. The leaderboard ceiling (0.9545) in this comp is a "blender artifact" —
     people share OOFs and average them. Your honest 0.953 is more
     transferable knowledge than their 0.9545.

WHAT TO LEARN NEXT
  • Probability calibration (isotonic, Platt) — relevant in deployment
  • Pseudo-labeling and semi-supervised tricks for tabular
  • Optuna hyperparameter optimization at scale
  • Production-style MLOps: experiment tracking (W&B, MLflow), feature stores
  • DAG-based pipelines: Hydra, Kedro, or DVC
  • Pre-trained tabular models more deeply (TabPFN-v2, TabTransformer, FT-Transformer)

CONGRATULATIONS — you took this from raw CSV to top-percent ensemble.
""")